In [1]:
# Useful Libraries
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, Sequential,Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing import image
import matplotlib.pyplot as plt
import numpy as np
import os

In [2]:
# Dataset Paths & Parameters
train_dir = "chest_xray/train"
val_dir = "chest_xray/val"
test_dir = "chest_xray/test"

IMG_SIZE = (224, 224)  # Input size for EfficientNet
BATCH_SIZE = 32        # Number of images per batch
EPOCHS = 10            # Number of training epochs

In [3]:
# Step 2: Preprocess & Augment Data
# Training data generator with augmentation
train_gen = ImageDataGenerator(
    rescale=1./255,          # Normalize pixels between 0 and 1
    rotation_range=20,       # Random rotations
    width_shift_range=0.1,   # Horizontal shift
    height_shift_range=0.1,  # Vertical shift
    shear_range=0.1,         # Shear transformations
    zoom_range=0.1,          # Random zoom
    horizontal_flip=True     # Flip images horizontally
)

# Validation & Test generator (no augmentation, only rescale)
val_test_gen = ImageDataGenerator(rescale=1./255)

# Load images from directories
train_data = train_gen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    color_mode='grayscale'
)

val_data = val_test_gen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    color_mode='grayscale'
)

test_data = val_test_gen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False,
    color_mode='grayscale'
)


Found 4894 images belonging to 2 classes.
Found 338 images belonging to 2 classes.
Found 624 images belonging to 2 classes.


In [4]:
# Model Building 
model = Sequential()

# Specify input shape separately
model.add(Input(shape=(224,224,1)))

# Now add layers without input_shape
model.add(layers.Conv2D(32, (3,3), activation='relu'))
model.add(layers.MaxPooling2D((2,2)))
model.add(layers.BatchNormalization())

# Rest of CNN
model.add(layers.Conv2D(64, (3,3), activation='relu'))
model.add(layers.MaxPooling2D((2,2)))
model.add(layers.BatchNormalization())

model.add(layers.Conv2D(128, (3,3), activation='relu'))
model.add(layers.MaxPooling2D((2,2)))
model.add(layers.BatchNormalization())

model.add(layers.Conv2D(256, (3,3), activation='relu'))
model.add(layers.MaxPooling2D((2,2)))
model.add(layers.BatchNormalization())

model.add(layers.Flatten())
model.add(layers.Dense(256, activation='relu'))
model.add(layers.Dropout(0.5))
model.add(layers.Dense(128, activation='relu'))
model.add(layers.Dropout(0.3))
model.add(layers.Dense(1, activation='sigmoid'))


In [5]:
# Compile the model
model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC()]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 222, 222, 32)        │             320 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 111, 111, 32)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 111, 111, 32)        │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 109, 109, 64)        │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 54, 54, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 54, 54, 64)          │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 52, 52, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 26, 26, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_2                │ (None, 26, 26, 128)         │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_3 (Conv2D)                    │ (None, 24, 24, 256)         │         295,168 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_3 (MaxPooling2D)       │ (None, 12, 12, 256)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_3                │ (None, 12, 12, 256)         │           1,024 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 36864)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 256)                 │       9,437,440 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 128)                 │          32,896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 1)                   │             129 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 9,860,225 (37.61 MB)

 Trainable params: 9,859,265 (37.61 MB)

 Non-trainable params: 960 (3.75 KB)

In [6]:
import warnings
warnings.filterwarnings('ignore')

# Train the model
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=EPOCHS
)

Epoch 1/10
153/153 ━━━━━━━━━━━━━━━━━━━━ 210936s 1388s/step - accuracy: 0.8388 - auc: 0.8906 - loss: 0.3748 - val_accuracy: 0.4822 - val_auc: 0.6419 - val_loss: 2.6873
Epoch 2/10
153/153 ━━━━━━━━━━━━━━━━━━━━ 291s 2s/step - accuracy: 0.8927 - auc: 0.9465 - loss: 0.2536 - val_accuracy: 0.4822 - val_auc: 0.5000 - val_loss: 3.8434
Epoch 3/10
153/153 ━━━━━━━━━━━━━━━━━━━━ 1067s 7s/step - accuracy: 0.9111 - auc: 0.9587 - loss: 0.2207 - val_accuracy: 0.4822 - val_auc: 0.8397 - val_loss: 2.3984
Epoch 4/10
153/153 ━━━━━━━━━━━━━━━━━━━━ 276s 2s/step - accuracy: 0.9185 - auc: 0.9631 - loss: 0.2073 - val_accuracy: 0.8314 - val_auc: 0.9782 - val_loss: 0.4250
Epoch 5/10
153/153 ━━━━━━━━━━━━━━━━━━━━ 274s 2s/step - accuracy: 0.9193 - auc: 0.9671 - loss: 0.1965 - val_accuracy: 0.9527 - val_auc: 0.9849 - val_loss: 0.2002
Epoch 6/10
153/153 ━━━━━━━━━━━━━━━━━━━━ 284s 2s/step - accuracy: 0.9238 - auc: 0.9720 - loss: 0.1830 - val_accuracy: 0.9704 - val_auc: 0.9918 - val_loss: 0.1253
Epoch 7/10
153/153 ━━━━━━━━

In [7]:
# Save the trained model
MODEL_PATH = "xray_cnn_model.h5"
model.save(MODEL_PATH)
print(f"Model saved as {MODEL_PATH}")

Model saved as xray_cnn_model.h5
